# Lab 2 — Aggregation과 인덱스 (40분)
기기 24대·mode_snapshot 3,360건·door_open 800건·door_close 799건의 합성 데이터를 사용합니다.
OPEN 횟수와 완료 구간 수를 구분하고 미완료 OPEN은 duration 평균에서 제외합니다.
MODE 코드는 의미 미확정 값이므로 door_open/door_close와 혼동하지 않습니다.
전용 DB `hrm_workshop_lab2`에서 적재(5분), 최신 조회·일별 분석(10분),
완료 시간·모델 분석(15분), explain 비교·리뷰(10분) 순으로 진행합니다.

## 공유 DB 실행 역할 — 먼저 확인하세요

| 표시 | 실행 담당 | 작업 |
|---|---|---|
| 🔴 강사 단독 실행 | 지정 강사 1명 | 적재·인덱스/컬렉션 생성·상태/알림 변경·임베딩 갱신 |
| 🟢 다중 실행 가능 | 모든 참가자 | find/aggregate/explain·건수/타입 확인·로컬 계산 |

**참가자:** 저장소 루트의 `Shared_DB_Participant.ipynb`에서 해당 Lab을 실행하세요.
이 파일의 강사 셀을 생략하면 변수 정의가 빠질 수 있으므로 참가자용은 독립된 읽기 코드로 구성했습니다.
**강사:** 아래 `INSTRUCTOR_MODE = True`로 변경한 뒤 준비/시연하세요. 참가자는 False를 유지합니다.
기본값 False에서는 쓰기 셀이 DB 변경 전에 중단됩니다. 표시된 이전 출력은 강사의 실행 예시입니다.

멱등 upsert도 DB 쓰기이므로 강사 전용입니다. "1회"는 **여러 사람이 실행하지 않는다는 의미**입니다.
재전송·부분 갱신·timer 테스트는 강사 한 명이 시나리오 순서대로 반복합니다.
Time Series insertMany·TTL insert·Kafka 재발행은 반복 시 중복/만료 변화가 발생할 수 있습니다.
공유 데이터 준비가 끝나면 강사가 완료를 안내하고 참가자 조회를 시작합니다.


In [ ]:
# 참가자는 False 유지. 지정 강사 한 명만 True로 변경합니다.
INSTRUCTOR_MODE = False

### 🔴 강사 단독 실행 · DB 변경
강사 준비·시연 단계에서만 실행합니다. 참가자 동시 실행 및 공유 DB Run All 금지.

In [1]:
# [강사 단독 실행] 공유 DB 변경은 지정된 강사 한 명만 수행합니다.
if not INSTRUCTOR_MODE:
    raise RuntimeError("강사 전용 셀입니다. 참가자는 Shared_DB_Participant.ipynb를 실행하세요.")
# 프로젝트 공통 모듈을 불러옵니다. 인증정보는 로컬 파일/환경변수/숨김 입력에서 읽습니다.
import sys
from pathlib import Path
from datetime import datetime, timedelta, timezone
from copy import deepcopy
from collections import Counter
from IPython.display import display
from bson import json_util
import pandas as pd
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lab2/events.ejson").exists())
sys.path.insert(0, str(root))
from workshop_utils import connect, load_samples, insert_once, table
UTC = timezone.utc

# 같은 파일을 재실행해도 결정적 _id 기준 원본 건수는 유지됩니다.
events, devices = load_samples()
client, db = connect("hrm_workshop_lab2")
insert_once(db.events, events)
insert_once(db.devices, devices)
history = db.events
assert history.count_documents({}) == 4959 and db.devices.count_documents({}) == 24
assert dict(Counter(e["eventType"] for e in events)) == {"mode_snapshot": 3360, "door_open": 800, "door_close": 799}
D = "LAB2-D01"
# 기간은 한국 시간 10월 11일 하루입니다. UTC로 변환한 시작 포함·끝 제외를 사용합니다.
START = datetime(2026, 10, 10, 15, tzinfo=UTC)
END = START + timedelta(days=1)
display(table(events[:2]))

연결 성공 / MongoDB 8.0.34 / DB hrm_workshop_lab2


,_id,deviceId,locationId,eventType,eventTime,receivedAt,schemaVersion,datasetTag,resource.href,resource.unitId,resource.component,context.modelCode,context.regionCode,context.countryCode,context.timeZone,payload.modes,notification.source,notification.capabilities
0,LAB2-D04-0-mode-0,LAB2-D04,LAB2-L04,mode_snapshot,2026-10-04 21:00:00+00:00,2026-10-04 21:00:00.300000+00:00,1,HRM_LAB2_DOOR_V1,/lab2/mode/0,0,main,LAB2-MODEL-1,DEMO-SEOUL,KR,Asia/Seoul,[MODE_01],SYNTHETIC_LAB2,[execute]
1,LAB2-D08-0-mode-0,LAB2-D08,LAB2-L08,mode_snapshot,2026-10-04 21:00:00+00:00,2026-10-04 21:00:00.300000+00:00,1,HRM_LAB2_DOOR_V1,/lab2/mode/0,0,main,LAB2-MODEL-2,DEMO-SEOUL,KR,Asia/Seoul,[MODE_01],SYNTHETIC_LAB2,[execute]


## 1. 리소스별 최신 이벤트
하나의 기기에서 모드와 도어가 서로 다른 리소스이므로 도어 리소스를 지정합니다.

### 🟢 다중 실행 가능 · 조회/로컬 계산
DB를 변경하지 않습니다. 이 노트북의 앞 셀 변수가 필요하면 참가자 전용 노트북을 사용하세요.

In [2]:
# 마지막 도어 이벤트가 미완료 OPEN인지 원본 이력으로 확인합니다.
latest_filter = {"deviceId": D, "resource.href": "/lab2/door/0"}
latest = list(history.find(latest_filter).sort([("eventTime", -1), ("_id", -1)]).limit(1))
assert latest[0]["_id"] == "LAB2-D01-open-unfinished" and latest[0]["payload"]["doorState"] == "open"
display(table(latest))

,_id,datasetTag,deviceId,eventTime,eventType,locationId,receivedAt,schemaVersion,context.modelCode,context.regionCode,context.countryCode,context.timeZone,notification.source,notification.capabilities,payload.doorState,resource.href,resource.unitId,resource.component
0,LAB2-D01-open-unfinished,HRM_LAB2_DOOR_V1,LAB2-D01,2026-10-11 12:00:00+00:00,door_open,LAB2-L01,2026-10-11 12:00:00.300000+00:00,1,LAB2-MODEL-1,DEMO-SEOUL,KR,Asia/Seoul,SYNTHETIC_LAB2,[execute],open,/lab2/door/0,0,main


## 2. 한국 시간의 일별·시간대별 OPEN 횟수
동일 기간의 Python 기준 계산과 MongoDB 집계를 비교합니다.

### 🟢 다중 실행 가능 · 조회/로컬 계산
DB를 변경하지 않습니다. 이 노트북의 앞 셀 변수가 필요하면 참가자 전용 노트북을 사용하세요.

In [3]:
# eventTime은 UTC BSON Date로 저장하고 일별 그룹에서만 한국 시간대를 적용합니다.
from zoneinfo import ZoneInfo
KST = ZoneInfo("Asia/Seoul")
open_filter = {"deviceId": D, "eventType": "door_open", "resource.href": "/lab2/door/0",
               "eventTime": {"$gte": START, "$lt": END}}
daily_pipeline = [{"$match": open_filter}, {"$group": {
    "_id": {"$dateToString": {"date": "$eventTime", "format": "%Y-%m-%d", "timezone": "Asia/Seoul"}},
    "openCount": {"$sum": 1}}}, {"$sort": {"_id": 1}}]
daily = list(history.aggregate(daily_pipeline))
selected = [e for e in events if e["deviceId"] == D and e["eventType"] == "door_open" and START <= e["eventTime"] < END]
expected = Counter(e["eventTime"].astimezone(KST).strftime("%Y-%m-%d") for e in selected)
assert {r["_id"]: r["openCount"] for r in daily} == dict(expected)
assert sum(r["openCount"] for r in daily) == 4
display(table(daily))
# 날짜와 시간을 같이 사용하여 서로 다른 날짜의 같은 시각을 잘못 합치지 않습니다.
hourly = [{"$match": open_filter}, {"$group": {
    "_id": {"$dateTrunc": {"date": "$eventTime", "unit": "hour", "timezone": "Asia/Seoul"}},
    "openCount": {"$sum": 1}}}, {"$sort": {"_id": 1}}]
display(table(history.aggregate(hourly)))

,_id,openCount
0,2026-10-11,4


,_id,openCount
0,2026-10-10 23:00:00+00:00,1
1,2026-10-11 03:00:00+00:00,1
2,2026-10-11 10:00:00+00:00,1
3,2026-10-11 12:00:00+00:00,1


## 3. 완료 구간의 합계·평균·최대
시작/종료가 모두 해당 기간 안에 있는 구간을 대상으로 합니다. 기간을 가로지르는 구간은 이 정의에서는 제외합니다.
elapsed 시간은 실제 Date 차이로 계산하며 미완료 구간을 0초로 넣지 않습니다.

### 🟢 다중 실행 가능 · 조회/로컬 계산
DB를 변경하지 않습니다. 이 노트북의 앞 셀 변수가 필요하면 참가자 전용 노트북을 사용하세요.

In [4]:
# CLOSE가 참조하는 OPEN과 동일 기기/리소스/시각인지 먼저 검증합니다.
by_id = {e["_id"]: e for e in events}
for e in events:
    if e["eventType"] == "door_close":
        o = by_id[e["payload"]["openEventId"]]
        assert o["deviceId"] == e["deviceId"] and o["resource"] == e["resource"]
        assert o["eventTime"] == e["payload"]["openedAt"] <= e["eventTime"]
duration_pipeline = [
    {"$match": {"deviceId": D, "eventType": "door_close", "resource.href": "/lab2/door/0",
                "eventTime": {"$gte": START, "$lt": END}, "payload.openedAt": {"$gte": START, "$lt": END}}},
    {"$set": {"durationSeconds": {"$divide": [{"$subtract": ["$eventTime", "$payload.openedAt"]}, 1000]}}},
    {"$group": {"_id": None, "closedCycles": {"$sum": 1}, "totalSeconds": {"$sum": "$durationSeconds"},
                "avgSeconds": {"$avg": "$durationSeconds"}, "maxSeconds": {"$max": "$durationSeconds"}}}
]
durations = list(history.aggregate(duration_pipeline))
assert durations == [{"_id": None, "closedCycles": 3, "totalSeconds": 300.0, "avgSeconds": 100.0, "maxSeconds": 240.0}]
display(table(durations))
print("OPEN 4회 / 완료 3구간 / 합계 300초 / 평균 100초 / 미완료 1구간")

,_id,closedCycles,totalSeconds,avgSeconds,maxSeconds
0,None,3,300.0,100.0,240.0


OPEN 4회 / 완료 3구간 / 합계 300초 / 평균 100초 / 미완료 1구간


## 4. 모델별 코호트 — 이벤트가 없는 기기도 포함
devices를 시작점으로 lookup하여 OPEN이 없는 기기를 0회로 유지합니다.
fullyObserved/observedDays는 fixture의 관측 계약이며 이벤트가 없다는 사실만으로 관측 완료를 추정하지 않습니다.

### 🟢 다중 실행 가능 · 조회/로컬 계산
DB를 변경하지 않습니다. 이 노트북의 앞 셀 변수가 필요하면 참가자 전용 노트북을 사용하세요.

In [5]:
# 기기 마스터가 분모입니다. 이벤트를 먼저 group하면 0회 기기가 빠질 수 있습니다.
cohort_pipeline = [
    {"$match": {"lab2.fullyObserved": True, "lab2.observedDays": 7}},
    {"$lookup": {"from": "events", "let": {"d": "$_id"}, "pipeline": [
        {"$match": {"eventType": "door_open", "eventTime": {"$gte": START, "$lt": END},
                    "$expr": {"$eq": ["$deviceId", "$$d"]}}}, {"$count": "n"}], "as": "openCounts"}},
    {"$set": {"openCount": {"$ifNull": [{"$first": "$openCounts.n"}, 0]}}},
    {"$group": {"_id": "$product.modelCode", "deviceCount": {"$sum": 1},
                "totalOpens": {"$sum": "$openCount"}, "avgOpensPerDevice": {"$avg": "$openCount"}}},
    {"$sort": {"_id": 1}}
]
cohort = list(db.devices.aggregate(cohort_pipeline))
expected_counts = Counter(e["deviceId"] for e in events if e["eventType"] == "door_open" and START <= e["eventTime"] < END)
for row in cohort:
    members = [d for d in devices if d["product"]["modelCode"] == row["_id"]]
    assert row["deviceCount"] == len(members)
    assert row["totalOpens"] == sum(expected_counts[d["_id"]] for d in members)
assert sum(r["deviceCount"] for r in cohort) == 24
display(table(cohort))

,_id,deviceCount,totalOpens,avgOpensPerDevice
0,LAB2-MODEL-1,8,39,4.875
1,LAB2-MODEL-2,8,42,5.250
2,LAB2-MODEL-3,8,49,6.125


## 5. explain — 같은 쿼리의 검사량 비교
이 fixture의 같은 조건·같은 결과를 COLLSCAN과 인덱스로 비교합니다.
재실행에도 `$natural` 힌트로 스캔 기준을 고정합니다.
작은 샘플의 시간 차이를 운영 성능으로 일반화하지 않고 docsExamined/keysExamined를 확인합니다.

### 🔴 강사 단독 실행 · DB 변경
강사 준비·시연 단계에서만 실행합니다. 참가자 동시 실행 및 공유 DB Run All 금지.

In [6]:
# [강사 단독 실행] 공유 DB 변경은 지정된 강사 한 명만 수행합니다.
if not INSTRUCTOR_MODE:
    raise RuntimeError("강사 전용 셀입니다. 참가자는 Shared_DB_Participant.ipynb를 실행하세요.")
# 재실행에도 같은 기준으로 비교하도록 전체 스캔은 $natural 힌트로 측정합니다.
query = {"find": history.name, "filter": open_filter,
         "sort": {"eventTime": -1}, "limit": 20}
scan = db.command("explain", {**query, "hint": {"$natural": 1}}, verbosity="executionStats")
history.create_index([("deviceId", 1), ("eventType", 1), ("resource.href", 1), ("eventTime", -1)],
                     name="device_type_resource_time")
indexed = db.command("explain", {**query, "hint": "device_type_resource_time"}, verbosity="executionStats")
rows = []
for label, plan in (("COLLSCAN baseline", scan), ("indexed", indexed)):
    stats = plan["executionStats"]
    rows.append({"strategy": label, **{k: stats[k] for k in
                ("nReturned", "totalDocsExamined", "totalKeysExamined", "executionTimeMillis")}})
assert rows[0]["nReturned"] == rows[1]["nReturned"] == 4
assert rows[1]["totalDocsExamined"] < rows[0]["totalDocsExamined"]
display(table(rows))
# 실제 반환 문서도 일치하는지 확인합니다. explain은 결과 문서를 반환하지 않습니다.
a = list(history.find(open_filter).sort("eventTime", -1).limit(20).hint([("$natural", 1)]))
b = list(history.find(open_filter).sort("eventTime", -1).limit(20).hint("device_type_resource_time"))
assert a == b
client.close()  # 노트북 종료 시 연결을 정리합니다.

,strategy,nReturned,totalDocsExamined,totalKeysExamined,executionTimeMillis
0,COLLSCAN baseline,4,4959,0,4
1,indexed,4,4,4,0


## 리뷰
- OPEN 4건과 완료 3구간의 차이를 설명하세요.
- 하루의 timezone/기간 경계 및 기간을 가로지르는 구간 정책을 정하세요.
- 중복·순서 역전·미완료 감지는 Lab 3에서, Time Series 비교는 Lab 4에서 확장합니다.
- 인덱스는 조회 패턴의 등호 조건 뒤 정렬/범위 조건을 기준으로 선택하고 쓰기 비용도 비교하세요.

[Aggregation](https://www.mongodb.com/docs/manual/aggregation/) ·
[Explain](https://www.mongodb.com/docs/manual/tutorial/analyze-query-plan/)